# Workspace Setup (Admin-only)

This notebook provisions shared resources for the course. Run it once per workspace before students begin.

**What it creates:**
1. Shared Unity Catalog namespace (`dbacademy.airbnb_agent`) with the Airbnb listings table
2. Read-only user permissions on all shared resources
3. A scheduled **Repair-Resources** job that recreates any shared resource that goes missing

Every step raises on failure, so the deployment stops at the first error and reports it.

**Alerting:** if a step fails, or the final check finds a resource missing, one message is posted to the Slack channel (`#training-automation-alert`) naming the failed step or the missing resources. Nothing is posted when setup succeeds. The Slack webhook URL is read from the Databricks secret `lab-alerts` / `slack-webhook` (see `_lib/slack_alert.py`); if the secret is missing, the alert is skipped with a warning and the run result is unchanged.

In [0]:
import sys, os
includes_dir = "."
config_version = "workspace-setup"
sys.path.insert(0, os.path.abspath('.'))
config_path = f"{includes_dir}/config/config-{config_version}.yaml"

from _lib.library_installer import install_libraries
install_libraries(config_path)

In [0]:
dbutils.library.restartPython()

In [0]:
import os

from databricks.sdk import WorkspaceClient

ws = WorkspaceClient()

# --- Slack alerting (see _lib/slack_alert.py) ---
import sys
from contextlib import contextmanager
sys.path.insert(0, os.path.abspath("."))
from _lib.slack_alert import send_slack_alert

COURSE_NAME = "Building Agentic Applications on Databricks"


@contextmanager
def setup_step(step):
    """Run one setup step. On failure, post a Slack alert naming the step, then re-raise so the run still fails."""
    try:
        yield
    except Exception as e:
        send_slack_alert(
            ws, dbutils,
            job_name="Workspace-Setup",
            summary=f"Workspace-Setup failed at step *{step}*. Shared lab resources may be incomplete.",
            course=COURSE_NAME,
            sections=[("Error", [f"{type(e).__name__}: {e}"])],
        )
        raise

## 1. Define Variables and Create Catalog / Schema

Set up the shared UC namespace and define all variable names used throughout this notebook. The `dbacademy` catalog must already exist.

In [0]:
# --- Shared UC namespace ---
SHARED_CATALOG = "dbacademy"
SHARED_SCHEMA  = "airbnb_agent"

# --- External (Marketplace) dependencies ---
# Loaded outside this notebook from the Marketplace share; Section 2 reads the
# listings CSV from the share's volume.
DATABRICKS_SHARE_NAME = "dbacademy_airbnb"
TABLE_NAME            = "sf_airbnb_listings"
TABLE_FULL_NAME       = f"{SHARED_CATALOG}.{SHARED_SCHEMA}.{TABLE_NAME}"
SOURCE_CSV_PATH       = f"/Volumes/{DATABRICKS_SHARE_NAME}/v01/sf-listings/sf-airbnb.csv"

# --- Repair job (keep in sync with Repair-Resources) ---
REPAIR_JOB_NAME      = "Repair Resources"
REPAIR_NOTEBOOK_NAME = "Repair-Resources"
REPAIR_CRON          = "0 */10 * * * ?"   # every 10 minutes, UTC (TESTING ONLY: set back before merging)

# Create catalog / schema
with setup_step("Create schema"):
    spark.sql(f"USE CATALOG {SHARED_CATALOG}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SHARED_SCHEMA}")
    spark.sql(f"USE SCHEMA {SHARED_SCHEMA}")

print(f"Shared catalog : {SHARED_CATALOG}")
print(f"Shared schema  : {SHARED_SCHEMA}")

## 2. Load the Airbnb Listings Table

Reads the listings CSV from the Marketplace share and writes the first 50 rows to `dbacademy.airbnb_agent.sf_airbnb_listings`.

In [0]:
from pyspark.sql.functions import concat, lit, col, regexp_replace, trim

with setup_step("Load listings table"):
    df = (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("multiLine", "true")
        .option("escape", '"')
        .load(SOURCE_CSV_PATH)
        .select("id", "name", "neighbourhood", "neighbourhood_cleansed", "summary", "price", "room_type")
        .withColumn("price", regexp_replace(trim(col("price")), "[$,]", "").cast("double"))
        .withColumn(
            "listing_source_information",
            concat(
                lit("ID of the property: "), col("id"),
                lit("\nName of the property: "), col("name"),
                lit("\nSummary of the property: "), col("summary"),
            ),
        )
        .limit(50)
    )

    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_FULL_NAME)
    print(f"Loaded {spark.table(TABLE_FULL_NAME).count()} rows into {TABLE_FULL_NAME}")

## 3. Grant read access to shared resources

In [0]:
# Grant read-only access on shared catalog resources
with setup_step("Grant permissions"):
    for stmt in [
        f"GRANT USE CATALOG ON CATALOG {SHARED_CATALOG} TO `account users`",
        f"GRANT USE SCHEMA ON SCHEMA {SHARED_CATALOG}.{SHARED_SCHEMA} TO `account users`",
        f"GRANT SELECT ON TABLE {TABLE_FULL_NAME} TO `account users`",
    ]:
        spark.sql(stmt)
        print(f"OK: {stmt}")

print(f"Granted read-only access on {SHARED_CATALOG}.{SHARED_SCHEMA} to all users")

## 4. Deploy Repair-Resources Job

Creates (or updates) a scheduled job that runs `Includes/Repair-Resources`. It checks every shared resource, recreates only what is missing, and fails if a repair does not work.

In [0]:
from databricks.sdk.service.jobs import CronSchedule, JobSettings, NotebookTask, PauseStatus, Task

with setup_step("Deploy Repair Resources job"):
    setup_path  = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    REPAIR_PATH = f"{os.path.dirname(setup_path)}/{REPAIR_NOTEBOOK_NAME}"

    job_settings = JobSettings(
        name=REPAIR_JOB_NAME,
        tasks=[Task(task_key="repair_resources", notebook_task=NotebookTask(notebook_path=REPAIR_PATH))],
        schedule=CronSchedule(quartz_cron_expression=REPAIR_CRON, timezone_id="UTC", pause_status=PauseStatus.UNPAUSED),
    )

    existing_jobs = list(ws.jobs.list(name=REPAIR_JOB_NAME))
    if existing_jobs:
        job_id = existing_jobs[0].job_id
        ws.jobs.reset(job_id=job_id, new_settings=job_settings)
        print(f"Updated existing job '{REPAIR_JOB_NAME}' (job_id={job_id})")
    else:
        job_id = ws.jobs.create(name=job_settings.name, tasks=job_settings.tasks, schedule=job_settings.schedule).job_id
        print(f"Created job '{REPAIR_JOB_NAME}' (job_id={job_id})")

    print(f"  Schedule: {REPAIR_CRON} (UTC)")
    print(f"  Notebook: {REPAIR_PATH}")

## 5. Verify Resources

Re-checks every resource. If anything is missing, posts a Slack alert listing it and raises, so the deployment is reported as failed.

In [0]:
missing = []


def _check(name, fn):
    try:
        ok, details = bool(fn()), ""
    except Exception as e:
        ok, details = False, f"{type(e).__name__}: {str(e)[:300]}"
    print(f"  [{'OK' if ok else 'MISSING'}] {name}" + (f" -> {details}" if details else ""))
    if not ok:
        missing.append(f"{name}: {details or 'not found'}")


def _has_grant(sec_type, sec_name, privilege):
    rows = spark.sql(f"SHOW GRANTS ON {sec_type} {sec_name}").collect()
    return any(r["Principal"] == "account users" and r["ActionType"] == privilege for r in rows)


_check(f"Catalog {SHARED_CATALOG}", lambda: ws.catalogs.get(SHARED_CATALOG))
_check(f"Schema {SHARED_CATALOG}.{SHARED_SCHEMA}", lambda: ws.schemas.get(f"{SHARED_CATALOG}.{SHARED_SCHEMA}"))
_check(f"Table {TABLE_FULL_NAME} (has rows)", lambda: spark.table(TABLE_FULL_NAME).limit(1).count() > 0)

for sec_type, sec_name, priv in [
    ("CATALOG", SHARED_CATALOG, "USE CATALOG"),
    ("SCHEMA", f"{SHARED_CATALOG}.{SHARED_SCHEMA}", "USE SCHEMA"),
    ("TABLE", TABLE_FULL_NAME, "SELECT"),
]:
    _check(f"{priv} ON {sec_type} {sec_name} -> account users",
           lambda t=sec_type, n=sec_name, p=priv: _has_grant(t, n, p))

_check(f"Job {REPAIR_JOB_NAME}", lambda: list(ws.jobs.list(name=REPAIR_JOB_NAME)))

if missing:
    send_slack_alert(
        ws, dbutils,
        job_name="Workspace-Setup",
        summary=f"Workspace-Setup finished but *{len(missing)}* shared resource(s) are missing.",
        course=COURSE_NAME,
        sections=[("Missing resources", missing)],
    )
    raise RuntimeError(
        f"Workspace-Setup verification failed: {len(missing)} resource(s) missing:\n"
        + "\n".join(f"  - {m}" for m in missing)
    )
print("All resources present.")

## Summary

In [0]:
print("=" * 60)
print("Workspace Setup Complete")
print("=" * 60)
print(f"  Shared catalog   : {SHARED_CATALOG}")
print(f"  Shared schema    : {SHARED_SCHEMA}")
print(f"  Table            : {TABLE_FULL_NAME}")
print(f"  Repair job       : {REPAIR_JOB_NAME} ({REPAIR_CRON} UTC)")
print(f"  User permissions : READ-ONLY on shared resources")